<table>
 <tr align=left><td><img align=left src="https://i.creativecommons.org/l/by/4.0/88x31.png">
 <td>Teks disediakan di bawah lisensi Creative Commons Attribution, CC-BY. Semua kode tersedia di bawah lisensi MIT yang disetujui FSF. (c) Kyle T. Mandli</td>
</table>

In [ ]:
from __future__ import print_function
%matplotlib inline
import numpy
import matplotlib.pyplot as plt

# Persamaan Hiperbolik - Bagian II

## Pelacakan Karakteristik

Cara umum untuk menyelesaikan PDB hiperbolik secara analitik adalah dengan menggunakan metode karakteristik, namun hingga saat ini kita belum benar-benar mencoba menggunakan teori ini untuk membangun metode numerik.  

Dengan memikirkan nilai solusi pada titik $(x_j, t + \Delta t)$, kita tahu untuk $a > 0$ bahwa kita melihat ke belakang ke titik $(x_j - a \Delta t, t)$ di mana solusi di sana memberikan informasi untuk solusi di titik yang diminati.

Hal ini bekerja dengan baik hingga kita mulai memikirkan tentang grid yang terdiskritisasi di mana kita hanya mengetahui solusi pada waktu $t$ pada sekumpulan titik diskrit.  

![Pelacakan Karakteristik nu != 1](./images/characteristic_tracing_1.png)

Jika karakteristik yang berpotongan dengan $(x_j, t + \Delta t)$ juga berpotongan dengan suatu titik pada waktu $t$ maka kita tidak masalah.  Biasanya hal ini tidak terjadi kecuali kita secara khusus memilih $\Delta x$ dan $\Delta t$ sedemikian sehingga kondisi ini terpenuhi.  Ternyata tentu saja hal ini terjadi jika
$$
    \frac{a \Delta t}{\Delta x} = 1,
$$
tepat batas atas dari hasil stabilitas kita.  

![Pelacakan Karakteristik nu == 1](./images/characteristic_tracing_2.png)

Demikian pula jika $\nu < 1$ maka kita tahu bahwa karakteristik tidak akan mengenai titik-titik grid secara tepat.  Perhatikan juga bahwa karena kendala $|\nu| \leq 1$ kita tahu bahwa karakteristik tidak dapat melewati $x_{j-1}$.

Kita juga bisa menginterpolasi antara dua nilai yang dipisahkan oleh karakteristik tersebut dan menemukan nilai yang dicari.  Tunjukkan bahwa melakukan hal ini dengan interpolasi linier menghasilkan metode upwind.

Untuk interpolasi linier kita tahu bahwa perpotongan berada di $x_p = x_j - a \Delta t$.  Interpolan linier adalah
$$\begin{aligned}
    P_1(x) &= \frac{x - x_{j-1}}{x_{j} - x_{j-1}} U^n_{j} + \frac{x - x_j}{x_{j-1} - x_j} U^n_{j-1} \\
    & = \frac{x - x_{j-1}}{\Delta x} U^n_{j} - \frac{x - x_j}{\Delta x} U^n_{j-1}
\end{aligned}$$
sehingga nilainya adalah
$$\begin{aligned}
    U^{n+1}_j = P_1(x_j - a \Delta t) &= \frac{x_j - a \Delta t - x_{j-1}}{\Delta x} U^n_{j} - \frac{x_j - a \Delta t - x_j}{\Delta x} U^n_{j-1} \\
    &= \frac{\Delta x - a \Delta t}{\Delta x} U^n_{j} + \frac{a \Delta t}{\Delta x} U^n_{j-1} \\
    &= U^n_{j} - \frac{a \Delta t}{\Delta x} (U^n_{j} - U^n_{j-1}).
\end{aligned}$$

Dengan teknik serupa kita juga dapat menemukan metode Beam-Warming dengan interpolasi kuadratik:
$$\begin{aligned}
    P_2(x) &= \frac{(x - x_{j-1})(x - x_{j-2})}{(x_{j} - x_{j-1}) (x_{j} - x_{j-2})} U^n_{j} + \frac{(x - x_{j})(x - x_{j-2})}{(x_{j-1} - x_{j}) (x_{j-1} - x_{j-2})} U^n_{j-1} + \frac{(x - x_{j})(x - x_{j-1})}{(x_{j-2} - x_{j}) (x_{j-2} - x_{j-1})} U^n_{j-2} \\
    &=\frac{(x - x_{j-1})(x - x_{j-2})}{2 \Delta x^2} U^n_{j} - \frac{(x - x_{j})(x - x_{j-2})}{\Delta x^2} U^n_{j-1} + \frac{(x - x_{j})(x - x_{j-1})}{2 \Delta x^2} U^n_{j-2} \\
    &=\frac{1}{\Delta x^2} \left[\frac{1}{2} U^n_{j} (x - x_{j-1})(x - x_{j-2}) - U^n_{j-1} (x - x_{j})(x - x_{j-2}) + \frac{1}{2} U^n_{j-2} (x - x_{j})(x - x_{j-1}) \right ]
\end{aligned}$$
dan akhirnya
$$\begin{aligned}
    U^{n+1}_j = P_2(x_j - a \Delta t) &= \frac{1}{\Delta x^2} \left[\frac{1}{2} U^n_{j} (\Delta x - a \Delta t)(2 \Delta x - a \Delta t) - U^n_{j-1} (- a \Delta t)(2 \Delta x - a \Delta t) + \frac{1}{2} U^n_{j-2} (- a \Delta t)(\Delta x - a \Delta t) \right ] \\
    &= \frac{1}{\Delta x^2} \left[\frac{1}{2} U^n_{j} (2 \Delta x^2 - 3 a \Delta t \Delta x + a \Delta t^2) - U^n_{j-1} (-2a \Delta t \Delta x + a^2 \Delta t^2) + \frac{1}{2} U^n_{j-2} (-a \Delta t \Delta x + a^2 \Delta t^2) \right ] \\
    &= U^n_j - \frac{a \Delta t}{2 \Delta x} (3 U^n_j - 4 U^n_{j-1} + U^n_{j-2}) + \frac{a \Delta t^2}{2 \Delta x^2} (U^n_j - 2 U^n_{j-1} + U^n_{j-2})
\end{aligned}$$

## Kondisi Courant-Friedrichs-Lewy (CFL)

Salah satu hasil menarik dari analisis karakteristik kita adalah bahwa kriteria stabilitas menyebabkan perpotongan karakteristik dengan $t_n$ berada dalam interval $[x_{j-1}, x_j]$ ketika $a > 0$.  Hal ini mengindikasikan prinsip yang lebih umum untuk stabilitas numerik PDB, yang dikemukakan oleh Courant, Friedrichs, dan Lewy dan sering disebut kondisi CFL.

Kondisi stabilitas yang telah kita amati berulang kali
$$
    \nu = \left | \frac{a \Delta t}{\Delta x} \right | \leq 1
$$
ternyata merupakan kondisi yang diperlukan untuk metode yang dikembangkan untuk menyelesaikan persamaan adveksi.  Nilai $\nu$ sering disebut *bilangan Courant* karena hal ini.

### Domain Ketergantungan

Untuk membuat pernyataan yang lebih umum tentang kondisi CFL dan bilangan Courant, kita perlu membicarakan apa itu *domain ketergantungan* untuk suatu PDB tertentu.  Kita sudah tahu apa ini untuk persamaan adveksi.  Kita tahu bahwa solusi pada $(X, T)$ adalah $u(X, T) = u_0(X - a T)$.  Domain ketergantungannya adalah
$$
    \mathcal{D}(X,T) = \{X - a T\}.
$$
Cara lain untuk memikirkan hal ini adalah dengan mempertimbangkan titik-titik mana yang bisa kita ubah yang akan mempengaruhi solusi pada $(X,T)$.  Dalam kasus persamaan adveksi, itu adalah satu titik.

Secara lebih umum untuk PDB lain kita mungkin mengharapkan domain ketergantungan lebih besar dari satu titik, melainkan berupa sekumpulan titik (seperti halnya pada sistem persamaan adveksi) atau seluruh interval.  Persamaan panas adalah salah satu persamaan seperti itu dan memiliki domain ketergantungan $\mathcal{D}(X, T) = (-\infty, \infty)$.  Dengan kata lain semua titik dalam domain mempengaruhi semua titik lain di setiap waktu mendatang.  Tipe persamaan ini juga dikatakan memiliki *kecepatan propagasi* tak hingga dan berlaku untuk setiap PDB parabolis serta merupakan cara lain untuk mengklasifikasikan PDB yang lebih kompleks.

Seseorang mungkin menolak gagasan ini untuk persamaan panas karena fungsi Green untuk titik tertentu meluruh secara eksponensial menjauhi suatu titik, namun sayangnya masih tidak cukup cepat.  Ini juga merupakan sumber kesimpulan dan keruntuhan fisik model difusi: material (atau panas) akan merambat dengan kecepatan tak terbatas.

### Domain Ketergantungan Numerik

Metode numerik juga memiliki domain ketergantungan yang ditentukan oleh stensil yang digunakan.  Misalnya metode Lax-Friedrichs memiliki solusi $U^n_j$ yang bergantung pada titik-titik $U^{n-1}_{j+1}$, $U^{n-1}_{j}$, dan $U^{n-1}_{j-1}$.  Hal ini umumnya berlaku untuk metode tiga titik yang kita kembangkan sebelumnya termasuk metode Lax-Wendroff.  

Kita juga dapat menelusuri ke belakang lebih jauh dalam waktu untuk melihat titik-titik mana yang bergantung pada $U^{n-1}_{j+1}$, $U^{n-1}_{j}$, dan $U^{n-1}_{j-1}$ sehingga terlihat kerucut ketergantungan yang terus melebar.

![Domain Ketergantungan](./images/characteristic_tracing_3.png)

Saat grid diperhalus baik dalam waktu maupun ruang dengan menghormati kriteria stabilitas (kondisi CFL), kita mungkin mengharapkan bahwa domain ketergantungan numerik akan konvergen ke yang sebenarnya.  Hal ini sebenarnya tidak berlaku untuk stensil tiga titik, tetapi kondisi yang lebih lemah memang berlaku, yaitu bahwa domain ketergantungan numerik harus mengandung domain ketergantungan PDB.

Jika kita terus memperhalus grid dengan rasio $\Delta t / \Delta x = r$ maka domain ketergantungan untuk titik $(X,T)$ akan mengisi interval $[X - T/ r, X+ T/r]$.  Karena kita ingin solusi yang dihitung $U(X,T)$ konvergen ke solusi sejati $u_0(X - a T)$, kita perlu mensyaratkan
$$
    X - T/r \leq X - a T \leq X + T /r.
$$
Hal ini pada dasarnya mengimplikasikan bahwa $u_0(X - aT)$ berada dalam kerucut ketergantungan numerik.  Ini juga mengimplikasikan bahwa $|a| \leq 1 /r$ dan oleh karena itu $|a \Delta t / \Delta x| \leq 1$ yang kembali memberikan kita kriteria stabilitas yang sudah dikenal.  Hal ini kemudian membawa kita pada pernyataan umum kondisi CFL.

Kondisi CFL dapat dirangkum dalam kondisi yang diperlukan berikut:
> Suatu metode numerik konvergen hanya jika domain ketergantungan numeriknya memuat domain ketergantungan yang ditentukan dari PDB asal ketika $\Delta t \rightarrow 0$ dan $\Delta x \rightarrow 0$.

### Contoh - Metode Upwind

Domain ketergantungan numerik bergantung pada tanda $a$ tetapi memiliki stensil 2 titik.  Perhatikan bahwa jika kita memilih arah yang salah untuk upwinding maka saat $\Delta t$ dan $\Delta x$ menuju 0, titik $X - a T$ tidak akan pernah berada dalam kerucut ketergantungan.

### Contoh - Persamaan Panas

Kita telah menyebutkan sebelumnya bahwa domain ketergantungan sejati untuk persamaan panas adalah seluruh domain.  Bagaimana hal itu bisa bekerja untuk persamaan panas, terutama dengan metode implisit?

Hal ini mengimplikasikan bahwa stensil 3-titik mana pun (yang selama ini kita gunakan) sebenarnya melanggar kondisi CFL.  Ini memang benar jika kita menetapkan rasio $\Delta t / \Delta x$, tetapi sebenarnya kita memiliki persyaratan yang lebih ketat untuk hubungan tersebut, yaitu $\Delta t / \Delta x^2 \leq 1 / 2$.  Hal ini memperluas domain ketergantungan saat $\Delta t \rightarrow 0$ cukup cepat sehingga akan mencakup seluruh domain.

Untuk metode implisit, seperti Crank-Nicholson, kondisi CFL terpenuhi untuk langkah waktu $\Delta t$ berapa pun karena adanya kopling antara setiap titik dengan titik lainnya.

## Persamaan Termodifikasi

Alat lain yang ampuh untuk menganalisis metode numerik adalah penggunaan persamaan termodifikasi.  Pendekatan ini serupa dengan yang kita gunakan untuk menurunkan galat pemotongan lokal dan mengungkapkan lebih banyak tentang bagaimana kita bisa mengharapkan metode numerik tertentu bekerja serta seperti apa galat tersebut.

Gagasan dasarnya adalah menemukan PDB baru yang dapat diselesaikan **secara tepat** oleh metode numerik.  Dengan kata lain jika kita memiliki PDB $v_t = f(v, v_x, v_{xx}, \cdots)$ maka solusi aproksimasi kita untuk suatu $\Delta t$ dan $\Delta x$ akan memenuhi $U^n_j = v(x_j, t_n)$.  Pertanyaannya juga bisa dirumuskan sebagai "apakah ada PDB yang lebih baik ditangkap oleh $U^n_j$?".  Kita dapat menjawab pertanyaan ini melalui ekspansi deret Taylor.

### Contoh - Metode Upwind

Metode upwind adalah
$$
    U^{n+1}_j = U^n_j - \frac{a \Delta t}{\Delta x} (U^n_j - U^n_{j-1}).
$$
dengan asumsi $a > 0$.  Asumsikan bahwa kita memiliki fungsi $v(x,t)$ dan PDB terkait (yang belum kita ketahui) yang diselesaikan secara tepat oleh metode upwind.

Pertama gantikan solusi diskrit $U$ dengan fungsi kontinu $v(x,t)$ sehingga kita memiliki
$$
    v(x, t + \Delta t) = v(x,t) - \frac{a \Delta t}{\Delta x} (v(x,t) - v(x-\Delta x,t)).
$$

Dengan menggunakan deret Taylor kita ketahui
$$\begin{aligned}
    \left(v + v_t \Delta t + \frac{\Delta t^2}{2} v_{tt} + \frac{\Delta t^3}{6} v_{ttt} + \cdots \right ) - v + \frac{a \Delta t}{\Delta x} \left( v - v + \Delta x v_x + \frac{\Delta x^2}{2} v_{xx} - \frac{\Delta x^3}{6} v_{xxx} + \cdots \right ) = 0\\    
    v_t + \frac{\Delta t}{2} v_{tt} + \frac{\Delta t^2}{6} v_{ttt} + \cdots + a \left( v_x + \frac{\Delta x}{2} v_{xx} - \frac{\Delta x^2}{6} v_{xxx} + \cdots \right ) = 0.
\end{aligned}$$
Dengan menyusun ulang suku-suku dalam persamaan kita memperoleh
$$
    v_t + a v_x = \frac{1}{2}(a \Delta x v_{xx} - \Delta t v_{tt}) + \frac{1}{6} (a \Delta x^2 v_{xxx} - \Delta t^2 v_{ttt}) + \cdots
$$
Inilah PDB yang dipenuhi oleh $v$.

Kita dapat melihat di sini bahwa jika $\Delta t$ dan $\Delta x$ mendekati nol kita dapat mengharapkan bahwa kita akan memulihkan persamaan asli yang dimaksudkan untuk diselesaikan oleh metode tersebut.  Suku-suku dominan di sisi kanan memberikan gambaran tentang perilaku solusi $v$ jika $\Delta t$ dan $\Delta x$ tidak nol.  Misalnya jika kita mempertimbangkan suku-suku $\mathcal{O}(\Delta x, \Delta t)$ kita memiliki persamaan
$$
    v_t + a v_x = \frac{1}{2}(a \Delta x v_{xx} - \Delta t v_{tt}),
$$
sebuah persamaan yang juga mencakup sesuatu yang terlihat seperti persamaan gelombang orde dua.  Kita dapat menulis ulang ini bahkan lebih eksplisit dengan mendiferensialkan kedua ruas terhadap $t$
$$
    v_{tt} = -a v_{xt} + \frac{1}{2} (a \Delta x v_{xxt} - \Delta t v_{ttt})
$$
dan terhadap $x$
$$
    v_{tx} = -a v_{xx} + \frac{1}{2} (a \Delta x v_{xxx} - \Delta t v_{ttx})
$$
yang digabungkan menghasilkan
$$
    v_{tt} = a^2 v_{xx} + \mathcal{O}(\Delta t).
$$

Mensubstitusikan kembali ke ekspresi asli di sisi kanan kita dapat menghilangkan turunan orde kedua dalam waktu untuk mendapatkan
$$
    v_t + a v_x = \frac{1}{2} a \Delta x \left(1 - \frac{a \Delta t}{\Delta x} \right) v_{xx} + \mathcal{O}(\Delta x^2, \Delta t^2)
$$
yang merupakan persamaan adveksi-difusi serupa dengan yang kita lihat sebelumnya tetapi kini dirumuskan secara eksplisit dalam kasus kontinu.  Kita juga dapat mengatakan bahwa diskritisasi upwind memberikan solusi persamaan adveksi-difusi di atas dengan akurasi orde dua.

Jadi apa yang dapat kita simpulkan dari ini?  
 - Perilaku orde utama ini membuat kita percaya bahwa galatnya akan bersifat difusif.
 - Jika $a \Delta t  = \Delta x$, yaitu bilangan Courant $\nu = 1$, maka solusi eksak akan diperoleh kembali.
 - Koefisien di depan operator difusi adalah $\frac{1}{2} (a \Delta x - a^2 \Delta t)$ yang positif jika $0 < a \Delta t / \Delta x < 1$, cara lain untuk melihat kriteria stabilitas.
 

### Contoh - Lax-Wendroff

Dengan mengikuti prosedur yang sama kita dapat menurunkan suku-suku orde utama (hingga $\mathcal{O}(\Delta t^2, \Delta x^2)$) untuk Lax-Wendroff dan menemukan
$$
    v_t + a v_x = -\frac{1}{6} a \Delta x^2 \left( 1 - \left(\frac{a \Delta t}{\Delta x}\right)^2 \right) v_{xxx}.
$$
Kita dapat mengamati beberapa hal dari persamaan termodifikasi ini
 - Lax-Wendroff mengaproksimasi persamaan adveksi-dispersi hingga orde tiga.
 - Galat dominan akan bersifat dispersif (turunan ketiga melakukan ini) meskipun galat ini akan lebih kecil daripada galat difusif dari metode upwind di atas.

#### Catatan Tambahan - Dispersi

Perhatikan PDB
$$
    u_t = u_{xxx}
$$
sebagai masalah Cauchy.  Jika kita melakukan transformasi Fourier terhadap persamaan tersebut kita tiba pada PDB
$$
    \hat{u~}_t(\xi,t) = - i \xi^3 \hat{u~}(\xi, t)
$$
yang memiliki solusi
$$
    \hat{u~}(\xi, t) = \hat{u~}_0(\xi) e^{-i \xi^3 t}.
$$

Perhatikan bahwa ini terlihat seperti solusi umum masalah adveksi dalam arti bahwa gelombang akan mempertahankan amplitudonya, namun setiap modus Fourier sekarang merambat dengan kecepatannya sendiri yang bergantung pada bilangan gelombangnya.  Kita dapat melihat ini dengan mengambil transformasi Fourier balik untuk mendapatkan
$$
    u(x,t) = \frac{1}{\sqrt{2 \pi}} \int^\infty_{-\infty} \hat{u~}_0(\xi) e^{i\xi(x - \xi^2 t)} d\xi.
$$

Dengan memeriksa integran kita dapat melihat bahwa bilangan gelombang $\xi$ merambat dengan kecepatan $\xi^2$.  Sebagai kontras, jalur serupa dengan persamaan adveksi menghasilkan
$$
    u(x,t) = \frac{1}{\sqrt{2 \pi}} \int^\infty_{-\infty} \hat{u~}_0(\xi) e^{i\xi(x - a t)} d\xi
$$
di mana kita jelas melihat semua bilangan gelombang $\xi$ merambat dengan kecepatan $a$.  Inilah perbedaan esensial antara adveksi dan dispersi: komponen-komponen solusi menyebar karena kecepatan efektifnya yang berbeda.

Kita dapat memperluas ini ke persamaan yang lebih umum berbentuk
$$
    u_t + a u_x + b u_{xxx} = 0
$$
di mana kita dapat menuliskan solusi
$$
    u(x,t) = \frac{1}{\sqrt{2 \pi}} \int^\infty_{-\infty} \hat{u~}_0(\xi) e^{i \xi (x - (a - b\xi^2) t)} d\xi.
$$
Di sini kita melihat kecepatan komponen merambat pada $a - b \xi^2$ sehingga nilai relatif $a$ dan $b$ akan menentukan efek mana yang akan lebih dominan.

Kembali ke persamaan termodifikasi metode Lax-Wendroff, kita dapat menuliskan kecepatan grup sebagai
$$
    c_g = a - \frac{1}{2} a \Delta x^2 \left(1 - \left( \frac{a \Delta t}{\Delta x} \right )^2 \right) \xi^2.
$$
Untuk metode ini $c_g < a$ untuk semua $\xi$ dan karenanya galat dispersi mengikuti gelombang sebagaimana terlihat dalam contoh numerik.

Kita juga dapat mempertahankan lebih banyak suku dalam persamaan termodifikasi, jika kita melakukan ini hingga orde keempat kita akan mendapatkan
$$
    v_t + a v_x + \frac{1}{6} a \Delta x^2 \left(1 - \left( \frac{a \Delta t}{\Delta x} \right )^2 \right) v_{xxx} + \epsilon v_{xxxx} = 0
$$
di mana $\epsilon = \mathcal{O}(\Delta x^3 + \Delta t^3)$.  Kita sekarang melihat bahwa melewati galat dispersif kita akan menemukan hiper-difusi sebagai galat utama.

Dispersi dan pembicaraan tentang bilangan gelombang $\xi$ juga menimbulkan pertimbangan penting lainnya.  Jika kita tertarik pada gelombang yang sangat berosilasi relatif terhadap grid, yaitu ketika $\xi \Delta x \gg 0$, kita mungkin menghadapi masalah dalam merepresentasikannya pada grid tertentu.  Untuk $\xi \Delta x$ yang cukup kecil hal ini bukanlah masalah dan persamaan termodifikasi memberikan perkiraan yang wajar mengenai dispersi dan karenanya kecepatan grup.  Jika solusi yang kita harapkan mengandung gelombang dengan $\xi \Delta x \gg 0$ maka suku-suku orde yang lebih tinggi mungkin diperlukan untuk merepresentasikan solusi dengan benar.  Biasanya kita mengandalkan substitusi ansatz 
$$
    u(x,t) = e^{i(\xi x_j - \omega(\xi) t_n)}.
$$
Hal ini jelas berkaitan dengan analisis von Neumann di mana kita telah menggantikan $g(\xi)$ dengan $e^{-i \omega(\xi) \Delta t}$.

### Contoh:  Beam-Warming

Sebagai kontras terhadap perilaku galat Lax-Wendroff, perhatikan persamaan termodifikasi untuk metode Beam-Warming yang adalah
$$
    v_t + a v_x = \frac{1}{6} a \Delta x^2 \left ( 2- \frac{3 a \Delta t}{\Delta x} + \left(\frac{a \Delta t}{\Delta x} \right)^2 \right ) v_{xxx}.
$$
Kita melihat dalam contoh numerik bahwa galat dispersi mendahului gelombang dan sekarang kita dapat melihat mengapa karena dalam kasus ini $c_g > a$.

### Contoh:  Leapfrog

Persamaan termodifikasi untuk leapfrog menghasilkan beberapa kesimpulan menarik karena adanya beberapa pembatalan yang menguntungkan.  Menulis metode leapfrog sebagai
$$
    \frac{v(x, t + \Delta t) - v(x, t - \Delta t)}{2 \Delta t} + a \frac{v(x + \Delta x, t) - v(x - \Delta x, t)}{2 \Delta x} = 0
$$
kita dapat mengamati bahwa persamaan termodifikasi berbentuk
$$
    v_t + a v_x + \frac{1}{6} a \Delta x^2 \left(1 - \left( \frac{a \Delta t}{\Delta x} \right )^2 \right) v_{xxx} = \epsilon_1 v_{xxxxx} + \epsilon_2 v_{xxxxxxx} + \cdots.
$$
Ternyata semua suku turunan orde genap gugur sehingga hanya menyisakan galat dispersif.  Bahkan hingga orde keempat, diskritisasi leapfrog menyelesaikan persamaan adveksi-dispersi.  Kita juga dapat melihat kembali mengapa leapfrog disebut non-dissipatif karena tidak ada suku galat yang memiliki turunan genap, yaitu difusi tidak hadir.

Sebagai latihan lebih lanjut kita juga dapat menghitung relasi dispersi tepat dari metode numerik (relasi dispersi menghubungkan bilangan gelombang $\xi$ dengan kecepatan fase, biasanya dilambangkan $\omega(\xi)$).  Mensubstitusikan ansatz yang dikenal mirip analisis von Neumann $e^{i(\xi x_j - \omega t_n)}$ kita memiliki
$$
    e^{-i\omega \Delta t} = e^{i \omega \Delta t} - \frac{a \Delta t}{\Delta x} \left( e^{i \xi \Delta x} - e^{-i\xi \Delta x}\right)
$$
yang menghasilkan
$$
    \sin(\omega \Delta t) = \frac{a \Delta t}{\Delta x} \sin(\xi \Delta x).
$$

Kita juga dapat menghitung kecepatan grup $c_g$ dari ini karena
$$
    c_g = \frac{\text{d} \omega}{\text{d} \xi} = \frac{a \cos(\xi \Delta x)}{\cos(\omega \Delta t)} = \pm \frac{a \cos(\xi \Delta x)}{\sqrt{1 - \nu^2 \sin^2(\xi \Delta x)}}.
$$
Perhatikan kembali apa yang terjadi jika $\nu = 1$.

## Sistem Persamaan Hiperbolik

Kita dapat memperluas apa yang telah kita lakukan sejauh ini ke sistem PDB hiperbolik (linear) berbentuk
$$
    u_t + A u_x = 0
$$
dengan kondisi awal yang sesuai $u(x,0) = u_0(x)$.  Di sini $A \in \mathbb R^{s \times s}$ di mana $s$ adalah jumlah persamaan.  

Dalam kasus ini terdapat cara yang terdefinisi dengan baik untuk memperluas gagasan sebelumnya kita tentang PDB hiperbolik karena kita mensyaratkan $A$ dapat didiagonalisasi dengan nilai eigen real agar sistem PDB disebut hiperbolik. Konsekuensinya adalah kita dapat menuliskan $A$ sebagai
$$
    A = R \Lambda R^{-1}
$$
di mana $R$ adalah vektor-vektor eigen dengan $\Lambda$ memuat nilai-nilai eigen pada diagonalnya.  Nilai-nilai eigen ini mengisi peran yang sebelumnya dimainkan oleh kecepatan advektif $a$ sehingga keberadaannya yang real dan terhingga sesuai dengan gagasan kita sebelumnya tentang apa yang seharusnya dimiliki persamaan hiperbolik, yaitu informasi merambat dengan kecepatan terhingga.

Meskipun kurang sederhana, kita masih dapat menyelesaikan sistem hiperbolik linear berkat dekomposisi $A$.  Mensubstitusikan dekomposisi tersebut dan mengalikan dengan $R^{-1}$ dari kanan menghasilkan
$$
    u_t + R \Lambda R^{-1} u_x = 0 \Rightarrow \\
    R^{-1} u_t + \Lambda R^{-1} u_x = 0.
$$

Mendefinisikan *variabel karakteristik* sebagai $w = R^{-1} u$ kita dapat menulis ulang sistem sebagai sekumpulan persamaan terdekopel dengan
$$
    w_t + \Lambda w_x = 0.
$$
Kita tahu cara menyelesaikan ini sebagai $w_p(x,t) = w_p(x - \lambda_p t, 0)$.  Kondisi awal dalam variabel karakteristik adalah
$$
    w(x, 0) = R^{-1} u_0(x).
$$

Kembali ke variabel asli, pada prinsipnya kita hanya perlu mengevaluasi
$$
    u(x,t) = R w(x,t)
$$
namun hal ini tidak mudah karena bentuk solusi dalam $w$.  Sebagai gantinya kita dapat menuliskan solusinya sebagai
$$
    u(x,t) = \sum^s_{p=1} w_p(x,t) r_p = \sum^s_{p=1}w_p(x - \lambda_p t, 0) r_p.
$$

Kita sekarang memiliki *karakteristik dari keluarga ke-$p$* yang merujuk pada kelompok karakteristik ke-$p$ yang ditentukan oleh nilai eigen ke-$p$.

### Metode Numerik

Kita dapat memperluas sebagian besar metode yang telah kita diskusikan sejauh ini ke sistem dengan cukup menggantikan kecepatan advektif $a$ dengan matriks $A$.  Sebagai contoh metode Lax-Wendroff dapat digeneralisasi menjadi
$$
     U^{n+1}_j = U^n_j - \frac{\Delta t}{2 \Delta x} A (U^n_{j+1} - U^n_{j-1})  + \frac{ \Delta t^2}{2 \Delta x^2} A^2 (U^n_{j+1} - 2 U^n_{j} + U^n_{j-1})
$$
asalkan bilangan Courant $\nu < 1$.  Perhatikan sekarang kita harus berhati-hati tentang bilangan Courant karena secara umum kondisi CFL mensyaratkan bahwa 
$$
    \nu = \max_{1 \leq p \leq s} \left| \frac{\lambda_p \Delta t}{\Delta x} \right | < 1
$$
Semua aproksimasi berbasis terpusat umumnya dapat diterapkan dengan kriteria stabilitas ini.

Metode yang kita pertimbangkan yang bersifat satu sisi memerlukan sedikit lebih banyak perhatian kecuali semua nilai eigen matriks $A$ bernilai positif atau negatif semua.  Sebaliknya kita harus menguraikan sistem ke dalam variabel karakteristiknya, menerapkan metode per persamaan, dan mentransformasi balik.  Umumnya jenis metode ini diklasifikasikan sebagai *metode Godunov*.

## Batas-Batas

Kita sebagian besar telah mengabaikan batas-batas selain yang bersifat periodik, jadi sekarang mari kita kembali ke pertanyaan tentang kondisi batas non-periodik. Sekarang mari kita pertimbangkan bagaimana memasukkan batas-batas untuk menemukan metode penyelesaian masalah nilai awal-batas.

Perhatikan sekarang PDB hiperbolik yang didefinisikan oleh
$$
    u_t + a u_x = 0 ~~~~ \Omega = [0, 1] \\
    u(x, 0) = u_0(x)
$$
sebelum mendefinisikan kondisi batas.  Berdasarkan diskusi domain ketergantungan kita sebelumnya, kita sedikit mengetahui kapan batas-batas akan mempengaruhi solusi kita.

![Batas karakteristik](./images/characteristics_regions_1.png)

Untuk persamaan skalar dan $a > 0$ maka kita tahu
$$
    u(x,t) = \left \{ \begin{aligned}
        u_0(x - a t) & & 0 \leq x - at \leq 1 \\
        g_0(t - x / a) & & \text{selain itu}.
    \end{aligned} \right .
$$

Jika kita memiliki sistem persamaan dengan tanda yang berlawanan untuk kecepatannya kita mungkin memiliki situasi yang terlihat seperti berikut ini
![Sistem PDB hiperbolik dengan batas-batas](./images/characteristics_regions_2.png)

### Upwind untuk IBVP

Katakanlah kita menggunakan metode upwind yang sesuai untuk $a > 0$ dengan grid $\Delta x = 1 / (m + 1)$.  Upwind mendeskripsikan semua persamaan internal dengan kondisi pada batas kiri yang memberikan nilai $U_0$, karenanya metode tersebut sepenuhnya menentukan masalah.  Metode ini stabil dengan kondisi yang sama seperti sebelumnya.

Perhatikan bahwa kita tidak dapat lagi menggunakan analisis von Neumann secara langsung karena batas yang baru.  Meskipun demikian hal ini masih dapat berguna sebagai alat stabilitas, tetapi analisis metode garis dapat lebih berguna di sini.

Perhatikan kembali sistem PDB
$$
    U'(t) = A U(t) + g(t)
$$
di mana
$$
    A = - \frac{a}{\Delta x} \begin{bmatrix}
        1 \\
        -1 & 1 \\
        & -1 & 1 \\
        & & \ddots & \ddots \\
        & & & -1 & 1
    \end{bmatrix} \quad g(t) = \begin{bmatrix} g_0(t) \frac{a}{\Delta x} \\ 0 \\ \vdots \\ 0 \end{bmatrix}.
$$

Sayangnya matriks baru ini, meskipun serupa dengan yang dipertimbangkan sebelumnya, memiliki sifat yang sangat berbeda.  Matriks baru ini memiliki nilai-nilai eigen yang terdistribusi secara seragam mengelilingi lingkaran dengan jari-jari $a / \Delta x$ dan berpusat di $z = - a / \Delta x$.

Mengapa perubahan-perubahan ini signifikan?  Jika kita mengikuti analisis sebelumnya kita akan menyimpulkan bahwa metode ini stabil jika
$$
    0 \leq \nu \leq 2
$$
yang agak mencurigakan.  Ternyata ini adalah kondisi yang diperlukan (meskipun jelas tidak cukup).  Masalah dalam analisis kita berasal dari fakta bahwa $A$ sangat non-normal dan kita memerlukan kendala lebih lanjut pada $\epsilon$-pseudospektra yang kembali mengarah pada kendala stabilitas yang lebih akrab bagi kita.

### Batas Aliran Keluar

Seperti yang disebutkan sebelumnya, sering kali metode numerik yang ingin kita gunakan akan memerlukan penggunaan kondisi batas di mana seharusnya tidak ada.  Kita melihat ini dengan metode Lax-Wendroff di mana titik-titik batas aliran keluar diperlukan oleh stensil.  Kita dapat menentukan *kondisi batas numerik* atau *kondisi batas artifisial* alih-alih menggunakan aproksimasi satu sisi.  Penentuan kondisi batas numerik cukup panjang dan analisisnya rumit sehingga di sini kita akan membatasi diri pada beberapa contoh ilustratif.

#### Contoh

Perhatikan metode leapfrog pada domain terbatas dengan $a > 0$ dan kondisi batas aliran masuk yang diberikan $g_0(t)$.  Katakanlah kita menggunakan metode upwind pada batas aliran keluar alih-alih menentukan kondisi.  Ternyata melakukan hal ini akan memperkenalkan gelombang dengan $\xi \Delta x \approx \pi$ yang akan bergerak ke kiri dengan kecepatan $-a$.

In [ ]:
# Implementasi Leapfrog untuk PDB u_t + u_x = 0 pada domain terbatas [0, 10]
# domain
u_true = lambda x, t: numpy.exp(-5.0 * ((x - t - 7.0)**2))

m = 100
x = numpy.linspace(0, 10.0, m)
delta_x = 10.0 / (m - 1)
cfl = 0.8
delta_t = cfl * delta_x

U = u_true(x, 0)
t = 0.0
# Mulai lompatan dengan solusi sejati
U_new = u_true(x, t + delta_t)
U_old = U_new.copy()

fig = plt.figure()
fig.set_figwidth(fig.get_figwidth() * 2)
fig.set_figheight(fig.get_figheight() * 3)
axes = fig.add_subplot(3, 2, 1)
axes.plot(x, U, 'ro')
axes.plot(x, u_true(x, t),'k')
axes.set_ylim((-0.1, 1.1))
axes.set_title("t = 0.0")

t += delta_t
for (n, t_final) in enumerate((10*delta_t, 50 * delta_t, 100 * delta_t, 200 * delta_t, 300 * delta_t)):
    while t < t_final:
        U_new[0] = U_old[0] - delta_t / delta_x * (U[1] - u_true(0.0, t))
        U_new[1:-1] = U_old[1:-1] - delta_t / delta_x * (U[2:] - U[:-2])
        # Gunakan upwind untuk batas aliran keluar
        U_new[-1] = U[-1] - delta_t / delta_x * (U[-1] - U[-2])
        U_old = U.copy()
        U = U_new.copy()
        t += delta_t

    # Tampilkan solusi pada t = 17.0 dan t = 0.0
    axes = fig.add_subplot(3, 2, n + 2)
    axes.plot(x, U, 'ro')
    axes.plot(x, u_true(x, t),'k')
    axes.set_ylim((-0.1, 1.1))
    axes.set_title("t = %s" % t_final)

plt.show()

Secara umum menangani kondisi batas aliran keluar sangat sulit.  Sering kali alih-alih menentukan metode satu sisi, kita ingin menentukan batas-batas numerik yang memiliki sifat khusus, seperti non-reflektif atau menyerap (kita melihat sedikit gelombang terpantul dalam contoh di atas).

## Alternatif

Akhirnya kita mengakhiri diskusi ini dengan beberapa alternatif yang tidak disebutkan di atas.

### Diskritisasi Orde Tinggi

Tentu saja kita dapat menggunakan diskritisasi orde tinggi yang sembarangan melebihi apa yang kita bicarakan di atas dengan menggunakan metode garis dan mendiskritisasi ruang dan waktu
$$
    U_j'(t) = -a W_j(t)
$$
dengan asumsi solusi tetap cukup mulus.  Satu contoh bisa jadi 
$$
    W_j(t) = \frac{4}{3} \left(\frac{U_{j+1} - U_{j-1}}{2 \Delta x} \right )- \frac{1}{3} \left(\frac{U_{j+2} - U_{j-2}}{4 \Delta x} \right ).
$$
Diskritisasi beda hingga yang telah dibahas sejauh ini semuanya dapat digunakan, tetapi akurasi orde tinggi yang kita capai datang dengan biaya stensil yang lebih lebar yang menyebabkan kesulitan karena alasan yang biasa.  

Pendekatan lain untuk menghindari hal ini adalah menggunakan metode pembedaan kompak yang menyelesaikan sistem linear.  Contoh sederhana dari gagasan ini adalah
$$
    \frac{1}{4} W_{j-1} + W_j + \frac{1}{4} W_{j+1} = \frac{3}{2} \left( \frac{U_{j+1} - U_{j-1}}{2 \Delta x} \right )
$$
yang menghasilkan aproksimasi $\mathcal{O}(\Delta x^4)$.

### Metode Spektral

Kita juga dapat menggunakan metode spektral untuk mentransformasi turunan-turunan spasial menjadi sistem linear.  Pada intinya kita dapat menurunkan matriks diferensiasi padat $D$ sehingga $W = D U$.  Metode ini dapat dengan mudah digeneralisasi ke sistem persamaan yang lebih kompleks tetapi memerlukan solusi yang mulus untuk bekerja dan bisa sangat sulit untuk dianalisis.

### Diskritisasi Waktu Lainnya

Tentu saja kita juga dapat menggunakan diskritisasi waktu yang berbeda.  Di atas kita menggunakan apa yang terlihat seperti forward Euler dan leapfrog tetapi Anda dapat menggunakan metode eksplisit orde tinggi seperti metode Runge-Kutta atau metode implisit.  Metode implisit dapat berguna jika Anda tidak terlalu khawatir tentang akurasi tetapi ingin mengembangkan solusi ke waktu yang besar.  Selain itu meskipun persamaan adveksi sendiri tidak kaku, beberapa PDB hiperbolik bisa jadi kaku atau diskritisasi spasial juga bisa demikian (seperti halnya dengan pendekatan spektral di atas).

### Hukum Kekekalan dan Metode Volume Hingga

Kelas besar dan penting dari PDB hiperbolik adalah hukum kekekalan berbentuk
$$
    u_t + f(u)_x = 0.
$$

Ini secara alami muncul di banyak bidang fisika dan menggambarkan evolusi besaran-besaran seperti massa, momentum, atau energi.  Salah satu sistem tersebut adalah persamaan Euler yang menggambarkan dinamika gas kompresibel:
$$\begin{aligned}
    &\rho_t + (\rho u)_x = 0 \\
    &(\rho u)_t + (\rho u^2 + p)_x = 0 \\
    &(E)_t + [(E + p) u]_x = 0
\end{aligned}$$
yang menggambarkan kerapatan $\rho$, momentum $\rho u$ dan energi $E$ yang digandeng dengan persamaan keadaan yang sesuai yang menghubungkan tekanan, kerapatan, dan energi internal.

Cara yang lebih alami untuk merumuskan hukum kekekalan adalah dengan bentuk integral dari persamaan yang sama.  Secara umum kita dapat menuliskan ini sebagai
$$
    \frac{\text{d}}{\text{d}t} \int^{x_2}_{x_1} u(x, t) dx = f(u(x_1,t)) - f(u(x_2, t)).
$$

Metode untuk menyelesaikan ini sering mengembangkan rata-rata sel dari $u$ daripada nilai titik.  Dalam hal ini aproksimasi kita $U^n_i$ dipandang sebagai rata-rata ini pada sel grid $[x_{i-1/2}, x_{i+1/2}]$ dengan panjang $\Delta x$ dan berpusat di $x_i$.  Rata-rata sel tersebut adalah
$$
    U^n_i \approx \frac{1}{\Delta x} \int^{x_{i+1/2}}_{x_{i-1/2}} u(x, t_n) dx.
$$
Metode volume hingga umumnya mengambil pendekatan ini dengan penentuan cara mengevaluasi fungsi-fungsi fluks sebagai tujuan numerik utama.